In [2]:
# ============================================================
# REBUILD CATBOOST V2 STACKING TABLES
# USING RAW CELLMINER RNA -> PCA50
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor

# -----------------------------
# PATHS
# -----------------------------

ROOT = Path("/Users/konuri/stacking")

PACKAGE = ROOT / "CATBOOST_TEAM_PACKAGE"

MASTER_PATH = Path(
    "/Users/konuri/model_final/data/almanac_final_59cell_104drug.csv"
)

RNA_PATH = Path(
    "/Users/konuri/TrustSyn/Generated_features/cellminer_features/cellminer_rna_features_59cell.csv"
)

FEATURE_LIST = PACKAGE / "CatBoost_62_feature_list.csv"


MODEL_ROOT = PACKAGE / "02_Final_Model"


OUT = PACKAGE


# -----------------------------
# LOAD MASTER
# -----------------------------

master = pd.read_csv(MASTER_PATH)

print("MASTER:", master.shape)


# -----------------------------
# LOAD CELLMINER RNA
# -----------------------------

rna = pd.read_csv(RNA_PATH)

print("RNA:", rna.shape)

cell_id = "cellminer_cellline_id"


# numeric genes only
gene_cols = [
    c for c in rna.columns
    if c != cell_id
]

rna_numeric = rna[gene_cols].astype(float)


# -----------------------------
# PCA 50
# -----------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(rna_numeric)


pca = PCA(
    n_components=50,
    random_state=42
)

pc = pca.fit_transform(X_scaled)


pc_cols = [
    f"CellMiner_PC{i}"
    for i in range(1,51)
]


cell_pc = pd.DataFrame(
    pc,
    columns=pc_cols
)


cell_pc[cell_id] = rna[cell_id]


print("PCA:", cell_pc.shape)


# -----------------------------
# FEATURE TABLE BUILDER
# -----------------------------

def build_features(test):

    df = (
        test
        .merge(
            cell_pc,
            on=cell_id,
            how="left"
        )
    )

    return df



# -----------------------------
# LOAD TEST SPLITS
# -----------------------------

splits = {

"RANDOM":
Path("/Users/konuri/model_final/splits/random/test.csv"),

"COLD_COMBINATION":
Path("/Users/konuri/model_final/splits/cold_combination/test.csv"),

"COLD_CELL":
Path("/Users/konuri/model_final/splits/cold_cell_line/test.csv")

}


# -----------------------------
# BUILD FEATURES
# -----------------------------

feature_cols = pd.read_csv(
    FEATURE_LIST
)["feature_name"].tolist()


print("\nFEATURE COUNT:",len(feature_cols))


for name,path in splits.items():

    print("\n====================")
    print(name)

    test = pd.read_csv(path)

    print("test:",test.shape)


    df = build_features(test)


    # add engineered features
    string = pd.read_csv(
        ROOT /
        "CATBOOST_TEAM_PACKAGE"/
        "COLD_DRUG_CatBoost_stacking_table.csv"
    )


    extra_cols = [
        "drug_A",
        "drug_B",
        "STRING_distance",
        "STRING_available",
        "KEGG_overlap",
        "Tanimoto_similarity",
        "target_count_A",
        "target_count_B",
        "shared_target_count",
        "union_target_count",
        "target_jaccard",
        "target_overlap_A",
        "target_overlap_B",
        "has_shared_target"
    ]


    extra = string[extra_cols].drop_duplicates(
        ["drug_A","drug_B"]
    )


    df = df.merge(
        extra,
        on=["drug_A","drug_B"],
        how="left"
    )


    X_test = df[feature_cols]


    print(
        "FINAL FEATURES:",
        X_test.shape
    )


    # -----------------------------
    # PREDICT ENSEMBLE
    # -----------------------------

    preds=[]


    models=list(
        (MODEL_ROOT/name).glob("*.cbm")
    )


    for model_path in models:

        model=CatBoostRegressor()

        model.load_model(
            model_path
        )

        preds.append(
            model.predict(X_test)
        )


    df["catboost_prediction"]=np.mean(
        preds,
        axis=0
    )


    df["y_true"]=test["combo_score"]


    output=OUT / (
        name+"_CatBoost_stacking_table.csv"
    )


    df.to_csv(
        output,
        index=False
    )


    print(
        "SAVED:",
        output
    )

MASTER: (294073, 9)
RNA: (59, 20203)
PCA: (59, 51)

FEATURE COUNT: 62

RANDOM
test: (29408, 9)
FINAL FEATURES: (29408, 62)
SAVED: /Users/konuri/stacking/CATBOOST_TEAM_PACKAGE/RANDOM_CatBoost_stacking_table.csv

COLD_COMBINATION
test: (29558, 9)
FINAL FEATURES: (29558, 62)
SAVED: /Users/konuri/stacking/CATBOOST_TEAM_PACKAGE/COLD_COMBINATION_CatBoost_stacking_table.csv

COLD_CELL
test: (29787, 9)
FINAL FEATURES: (29787, 62)
SAVED: /Users/konuri/stacking/CATBOOST_TEAM_PACKAGE/COLD_CELL_CatBoost_stacking_table.csv


In [3]:
from pathlib import Path
import pandas as pd

folder = Path("/Users/konuri/stacking/CATBOOST_TEAM_PACKAGE")

for f in [
    "RANDOM_CatBoost_stacking_table.csv",
    "COLD_COMBINATION_CatBoost_stacking_table.csv",
    "COLD_CELL_CatBoost_stacking_table.csv"
]:
    df = pd.read_csv(folder/f)
    
    print("\n",f)
    print(df.shape)
    print("y_true:", "y_true" in df.columns)
    print("prediction:", "catboost_prediction" in df.columns)
    print(df[["combo_score","y_true","catboost_prediction"]].head())


 RANDOM_CatBoost_stacking_table.csv
(29408, 73)
y_true: True
prediction: True
   combo_score    y_true  catboost_prediction
0    -0.333333 -0.333333            -1.731061
1     1.222222  1.222222            -0.114627
2     6.666667  6.666667            -0.183730
3    -2.555556 -2.555556            -0.776740
4    -4.000000 -4.000000            -0.781407

 COLD_COMBINATION_CatBoost_stacking_table.csv
(29558, 73)
y_true: True
prediction: True
   combo_score    y_true  catboost_prediction
0     6.333333  6.333333            -1.548757
1     3.666667  3.666667            -0.891503
2    -8.555556 -8.555556            -0.905059
3    -1.333333 -1.333333            -1.112232
4    -7.555556 -7.555556             0.020808

 COLD_CELL_CatBoost_stacking_table.csv
(29787, 73)
y_true: True
prediction: True
   combo_score    y_true  catboost_prediction
0     1.222222  1.222222            -1.528834
1     4.555556  4.555556             0.577298
2    -1.555556 -1.555556            -1.647917
3     8.777778

In [4]:
from pathlib import Path

for p in Path("/Users/konuri").rglob("*prediction*.csv"):
    print(p)

/Users/konuri/xgboost_trustsyn/xgboost_baseline_6A_predictions.csv
/Users/konuri/model_final/DMPNN_results/baseline_analysis/random_test_predictions.csv
/Users/konuri/model_final/cuda_resume_outputs/predictions/random_validation_predictions_cuda.csv
/Users/konuri/model_final/cuda_resume_outputs/predictions/random_test_predictions_cuda.csv
/Users/konuri/Downloads/miniconda3/envs/FDS/lib/python3.13/site-packages/statsmodels/gam/tests/results/prediction_from_mgcv.csv
/Users/konuri/stacking/CATBOOST_TEAM_PACKAGE/COLD_DRUG_CatBoost_predictions.csv
/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/COLD_CELL_LINE/final_v2_c_test_predictions.csv
/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/COLD_COMBINATION/final_v2_c_test_predictions.csv
/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/RANDOM/final_v2_c_test_predictions.csv
/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/COLD_DRUG/final_v2_c_test_predictions.csv
/Users/konuri/stacking/TrustSyn_STACKED_RESULTS/COLD_DRUG/COLD_DRUG_stacke

In [5]:
import pandas as pd
from pathlib import Path

files = {
"RANDOM":
"/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/RANDOM/final_v2_c_test_predictions.csv",

"COLD_COMBINATION":
"/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/COLD_COMBINATION/final_v2_c_test_predictions.csv",

"COLD_CELL":
"/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/COLD_CELL_LINE/final_v2_c_test_predictions.csv"
}


for name,path in files.items():

    df=pd.read_csv(path)

    print("\n================")
    print(name)
    print(df.shape)
    print(df.columns.tolist())
    print(df.head())


RANDOM
(29408, 11)
['drug_A', 'drug_B', 'CELLNAME', 'tissue', 'combo_score', 'CONC1', 'CONC2', 'cellminer_cellline_id', 'nci_almanac_cellname', 'target', 'prediction']
     drug_A    drug_B  CELLNAME          tissue  combo_score         CONC1  \
0   38721.0   82151.0   SK-OV-3  Ovarian Cancer    -0.333333  7.400000e-06   
1  754143.0  755986.0    SF-539      CNS Cancer     1.222222  3.700000e-09   
2  125066.0  763371.0        SR        Leukemia     6.666667  3.700000e-08   
3  109724.0  686673.0    SF-295      CNS Cancer    -2.555556  3.700000e-05   
4   63878.0  761432.0  HCC-2998    Colon Cancer    -4.000000  1.850000e-07   

          CONC2 cellminer_cellline_id nci_almanac_cellname    target  \
0  7.400000e-08            OV:SK-OV-3              SK-OV-3 -0.333333   
1  9.250000e-06            CNS:SF-539                  NaN  1.222222   
2  3.700000e-06                 LE:SR                   SR  6.666667   
3  7.400000e-06            CNS:SF-295                  NaN -2.555556   
4 

In [6]:
import pandas as pd
from pathlib import Path


CAT_DIR = Path(
    "/Users/konuri/stacking/CATBOOST_TEAM_PACKAGE"
)

DMPNN_DIR = Path(
    "/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS"
)

OUT_DIR = Path(
    "/Users/konuri/stacking/STACKING_TABLES"
)

OUT_DIR.mkdir(exist_ok=True)


files = {

"RANDOM": (
    CAT_DIR/"RANDOM_CatBoost_stacking_table.csv",
    DMPNN_DIR/"RANDOM/final_v2_c_test_predictions.csv"
),

"COLD_COMBINATION": (
    CAT_DIR/"COLD_COMBINATION_CatBoost_stacking_table.csv",
    DMPNN_DIR/"COLD_COMBINATION/final_v2_c_test_predictions.csv"
),

"COLD_CELL": (
    CAT_DIR/"COLD_CELL_CatBoost_stacking_table.csv",
    DMPNN_DIR/"COLD_CELL_LINE/final_v2_c_test_predictions.csv"
)

}


keys = [
    "drug_A",
    "drug_B",
    "CELLNAME"
]


for name,(cat_path,dmpnn_path) in files.items():

    print("\n================")
    print(name)


    cat = pd.read_csv(cat_path)

    dmpnn = pd.read_csv(dmpnn_path)


    dmpnn_small = dmpnn[
        keys + ["prediction"]
    ].rename(
        columns={
            "prediction":"dmpnn_prediction"
        }
    )


    stack = cat.merge(
        dmpnn_small,
        on=keys,
        how="inner",
        validate="one_to_one"
    )


    print("CAT:",cat.shape)
    print("DMPNN:",dmpnn.shape)
    print("STACK:",stack.shape)


    print(
        "Missing DMPNN:",
        stack.dmpnn_prediction.isna().sum()
    )


    out = OUT_DIR / (
        name+"_STACKING_TABLE.csv"
    )


    stack.to_csv(
        out,
        index=False
    )


    print("SAVED:",out)


RANDOM
CAT: (29408, 73)
DMPNN: (29408, 11)
STACK: (29408, 74)
Missing DMPNN: 0
SAVED: /Users/konuri/stacking/STACKING_TABLES/RANDOM_STACKING_TABLE.csv

COLD_COMBINATION
CAT: (29558, 73)
DMPNN: (29558, 11)
STACK: (29558, 74)
Missing DMPNN: 0
SAVED: /Users/konuri/stacking/STACKING_TABLES/COLD_COMBINATION_STACKING_TABLE.csv

COLD_CELL
CAT: (29787, 73)
DMPNN: (29787, 11)
STACK: (29787, 74)
Missing DMPNN: 0
SAVED: /Users/konuri/stacking/STACKING_TABLES/COLD_CELL_STACKING_TABLE.csv


In [8]:
from pathlib import Path

for p in Path("/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS").rglob("*validation*"):
    print(p)

In [9]:
from pathlib import Path

root = Path("/Users/konuri/stacking/TrustSyn_DMPNN")

for p in root.rglob("*.csv"):
    if any(x in p.name.lower() for x in [
        "val",
        "valid",
        "validation",
        "dev"
    ]):
        print(p)

In [10]:
df = pd.read_csv(
"/Users/konuri/stacking/STACKING_TABLES/RANDOM_STACKING_TABLE.csv"
)

print(df.columns.tolist())

['drug_A', 'drug_B', 'CELLNAME', 'tissue', 'combo_score', 'CONC1', 'CONC2', 'cellminer_cellline_id', 'nci_almanac_cellname', 'CellMiner_PC1', 'CellMiner_PC2', 'CellMiner_PC3', 'CellMiner_PC4', 'CellMiner_PC5', 'CellMiner_PC6', 'CellMiner_PC7', 'CellMiner_PC8', 'CellMiner_PC9', 'CellMiner_PC10', 'CellMiner_PC11', 'CellMiner_PC12', 'CellMiner_PC13', 'CellMiner_PC14', 'CellMiner_PC15', 'CellMiner_PC16', 'CellMiner_PC17', 'CellMiner_PC18', 'CellMiner_PC19', 'CellMiner_PC20', 'CellMiner_PC21', 'CellMiner_PC22', 'CellMiner_PC23', 'CellMiner_PC24', 'CellMiner_PC25', 'CellMiner_PC26', 'CellMiner_PC27', 'CellMiner_PC28', 'CellMiner_PC29', 'CellMiner_PC30', 'CellMiner_PC31', 'CellMiner_PC32', 'CellMiner_PC33', 'CellMiner_PC34', 'CellMiner_PC35', 'CellMiner_PC36', 'CellMiner_PC37', 'CellMiner_PC38', 'CellMiner_PC39', 'CellMiner_PC40', 'CellMiner_PC41', 'CellMiner_PC42', 'CellMiner_PC43', 'CellMiner_PC44', 'CellMiner_PC45', 'CellMiner_PC46', 'CellMiner_PC47', 'CellMiner_PC48', 'CellMiner_PC49', 'C

In [11]:
from pathlib import Path
import pandas as pd

# ============================================================
# SEARCH FOR EXISTING TRAIN / OOF PREDICTIONS
# ============================================================

SEARCH_DIRS = [
    Path("/Users/konuri/stacking"),
    Path("/Users/konuri/model_final"),
]

keywords = [
    "train_predictions",
    "train_prediction",
    "oof",
    "OOF",
    "validation_predictions",
    "val_predictions"
]

print("Searching for existing meta-training predictions...\n")

found = []

for base in SEARCH_DIRS:
    for f in base.rglob("*"):
        if f.is_file():
            name = f.name.lower()
            if any(k.lower() in name for k in keywords):
                found.append(f)

for f in found:
    print(f)

print("\nTOTAL FOUND:", len(found))

Searching for existing meta-training predictions...

/Users/konuri/model_final/cuda_resume_outputs/predictions/random_validation_predictions_cuda.csv

TOTAL FOUND: 1


In [14]:
import pandas as pd

for f in [
"/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/RANDOM/final_v2_c_test_predictions.csv",
"/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/COLD_COMBINATION/final_v2_c_test_predictions.csv",
"/Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/COLD_CELL_LINE/final_v2_c_test_predictions.csv"
]:
    df = pd.read_csv(f)
    print("\n", f)
    print(df.columns.tolist())


 /Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/RANDOM/final_v2_c_test_predictions.csv
['drug_A', 'drug_B', 'CELLNAME', 'tissue', 'combo_score', 'CONC1', 'CONC2', 'cellminer_cellline_id', 'nci_almanac_cellname', 'target', 'prediction']

 /Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/COLD_COMBINATION/final_v2_c_test_predictions.csv
['drug_A', 'drug_B', 'CELLNAME', 'tissue', 'combo_score', 'CONC1', 'CONC2', 'cellminer_cellline_id', 'nci_almanac_cellname', 'target', 'prediction']

 /Users/konuri/stacking/TrustSyn_DMPNN/PREDICTIONS/COLD_CELL_LINE/final_v2_c_test_predictions.csv
['drug_A', 'drug_B', 'CELLNAME', 'tissue', 'combo_score', 'CONC1', 'CONC2', 'cellminer_cellline_id', 'nci_almanac_cellname', 'target', 'prediction']


In [15]:
from pathlib import Path

for split in [
    "random",
    "cold_combination",
    "cold_cell_line"
]:
    print("\n================")
    print(split.upper())

    base = Path("/Users/konuri/model_final/splits") / split

    for f in ["train.csv", "val.csv", "test.csv"]:
        p = base / f
        print(f, p.exists(), p)


RANDOM
train.csv True /Users/konuri/model_final/splits/random/train.csv
val.csv True /Users/konuri/model_final/splits/random/val.csv
test.csv True /Users/konuri/model_final/splits/random/test.csv

COLD_COMBINATION
train.csv True /Users/konuri/model_final/splits/cold_combination/train.csv
val.csv True /Users/konuri/model_final/splits/cold_combination/val.csv
test.csv True /Users/konuri/model_final/splits/cold_combination/test.csv

COLD_CELL_LINE
train.csv True /Users/konuri/model_final/splits/cold_cell_line/train.csv
val.csv True /Users/konuri/model_final/splits/cold_cell_line/val.csv
test.csv True /Users/konuri/model_final/splits/cold_cell_line/test.csv


In [16]:
from pathlib import Path

for f in Path("/Users/konuri/stacking/TrustSyn_DMPNN").rglob("*"):
    if f.suffix in [".py", ".ipynb"]:
        print(f)

/Users/konuri/stacking/TrustSyn_DMPNN/FINAL_MODEL/Untitled1.ipynb
/Users/konuri/stacking/TrustSyn_DMPNN/FINAL_MODEL/CatBoost_DMPNN_Stacking.ipynb
/Users/konuri/stacking/TrustSyn_DMPNN/FINAL_MODEL/Untitled.ipynb
/Users/konuri/stacking/TrustSyn_DMPNN/FINAL_MODEL/Model_DMPNN.ipynb
/Users/konuri/stacking/TrustSyn_DMPNN/FINAL_MODEL/.ipynb_checkpoints/Model_DMPNN-checkpoint.ipynb
/Users/konuri/stacking/TrustSyn_DMPNN/FINAL_MODEL/.ipynb_checkpoints/Untitled1-checkpoint.ipynb
/Users/konuri/stacking/TrustSyn_DMPNN/FINAL_MODEL/.ipynb_checkpoints/Untitled-checkpoint.ipynb
/Users/konuri/stacking/TrustSyn_DMPNN/FINAL_MODEL/.ipynb_checkpoints/CatBoost_DMPNN_Stacking-checkpoint.ipynb


In [ ]:
# ============================================================
# CATBOOST PREDICTION LOOP WITH FEATURE BUILDING
# ============================================================

for split_name, folder in splits.items():

    print("\n================")
    print(split_name)

    split_path = SPLIT_DIR / folder


    for part in ["train", "val", "test"]:

        csv = split_path / f"{part}.csv"

        df = pd.read_csv(csv)

        print("\n", part, df.shape)


        # IMPORTANT:
        # convert raw 9 columns -> 65 engineered columns
        df_features = build_features(df)


        print(
            "after feature build:",
            df_features.shape
        )


        # keep only CatBoost 62 features
        X = df_features[features]


        print(
            "CatBoost input:",
            X.shape
        )


        pred = catboost_predict(
            split_name,
            X
        )


        out = df.copy()

        out["catboost_prediction"] = pred


        save = (
            OUTPUT_DIR /
            f"{split_name}_{part}_CATBOOST_ONLY.csv"
        )


        out.to_csv(
            save,
            index=False
        )


        print(
            "saved:",
            save
        )


print("\nDONE")


RANDOM

 train (235258, 9)
